# Vision Transformer (ViT)
"An Image is Worth 16x16 Words" (Dosovitskiy et al., 2020)

**Idea:** cut the image into fixed-size patches, treat each patch as a token, and feed the sequence to a standard Transformer **encoder**.

### ViT vs original Transformer (Vaswani et al., 2017)
| | Transformer (NLP) | ViT |
|---|---|---|
| Input tokens | word ids -> `nn.Embedding` lookup | image patches -> linear projection (a conv with kernel = stride = patch size) |
| Parts used | encoder + decoder | **encoder only** (like BERT) |
| Positional encoding | fixed sin/cos | **learned** `nn.Parameter` |
| Norm placement | Post-LN: `x = LN(x + f(x))` | **Pre-LN**: `x = x + f(LN(x))` (trains more stably) |
| FFN activation | ReLU | GELU |
| Attention mask | causal mask in decoder | **no mask**, every patch attends to every patch |
| Output | next-token distribution per position | one class from the `[CLS]` token |

Shape flow: `(N, 3, 32, 32) -> patches (N, 64, 48) -> embed (N, 64, D) -> +[CLS] (N, 65, D) -> encoder x L -> CLS (N, D) -> logits (N, 10)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Patch embedding
Transformer: `token id -> embedding row`.  ViT: `patch pixels -> linear layer`.

A `Conv2d` with `kernel_size = stride = patch_size` is exactly "flatten each patch and apply the same `nn.Linear`" - just faster.

### Formula: Patch embedding
$$z_0 = [\,x_{\text{cls}};\ x_p^1 E;\ x_p^2 E;\ \dots;\ x_p^N E\,] + E_{pos}, \qquad N = \frac{HW}{P^2}$$
- $x \in \mathbb{R}^{H \times W \times C}$: input image (here $32 \times 32 \times 3$)
- $P$: patch size (here 4); $C$: channels (3); $D$: embedding dim (192)
- $N$: number of patches ($32 \cdot 32 / 4^2 = 64$)
- $x_p^i \in \mathbb{R}^{P^2 C}$: $i$-th patch, flattened (48 values)
- $E \in \mathbb{R}^{(P^2 C) \times D}$: shared linear patch projection (the `Conv2d`)
- $x_{\text{cls}} \in \mathbb{R}^{D}$: learned `[CLS]` token
- $E_{pos} \in \mathbb{R}^{(N+1) \times D}$: learned positional embedding
- $[\,\cdot\,;\,\cdot\,]$: concatenation along the sequence axis; $z_0 \in \mathbb{R}^{(N+1) \times D}$

In [ ]:
class PatchEmbedding(nn.Module):
    # (N, C, H, W) -> (N, num_patches, D)
    # e.g. (N, 3, 32, 32), patch 4 -> 8x8 = 64 patches -> (N, 64, D)
    def __init__(self, img_size, patch_size, in_ch, dim):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                 # (N, D, 8, 8)
        x = x.flatten(2).transpose(1, 2) # (N, 64, D)
        return x

## 2. Multi-head self-attention
Same as the Transformer (scaled dot-product attention, formula below).

Only difference: **no causal mask** (it's an encoder, every patch can see every other patch).

### Formula: Scaled dot-product attention
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V, \qquad Q = XW_Q,\ K = XW_K,\ V = XW_V$$
- $X \in \mathbb{R}^{T \times D}$: input tokens ($T = N+1 = 65$)
- $W_Q, W_K, W_V \in \mathbb{R}^{D \times d_k}$: learned projections (fused in `self.qkv`)
- $Q, K, V$: queries, keys, values, each $T \times d_k$
- $d_k$: per-head dim ($D/h = 64$)
- $QK^T \in \mathbb{R}^{T \times T}$: similarity of every token pair
- $\sqrt{d_k}$: dot products of $d_k$ unit-variance terms have variance $d_k$; dividing keeps scores at variance ~1 so softmax doesn't saturate (vanishing gradients)
- softmax is applied row-wise; no mask (encoder)

### Formula: Multi-head attention
$$\text{MSA}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W_O, \qquad \text{head}_i = \text{Attention}(XW_Q^i, XW_K^i, XW_V^i)$$
- $h$: number of heads (3)
- $W_Q^i, W_K^i, W_V^i$: projections of head $i$
- Concat: stacks head outputs back to $T \times h d_k = T \times D$
- $W_O \in \mathbb{R}^{D \times D}$: output projection (`self.proj`)

### Formula: Softmax
$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$$
- $x \in \mathbb{R}^n$: vector of scores (logits)
- output: non-negative, sums to 1 (a probability distribution)

In [ ]:
class MultiHeadAttention(nn.Module):
    # (N, T, D) -> (N, T, D)
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim)   # Q, K, V in one matmul
        self.proj = nn.Linear(dim, dim)      # output projection W_O
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        N, T, D = x.shape
        qkv = self.qkv(x).reshape(N, T, 3, self.num_heads, self.head_dim)  # (N, T, 3D) -> (N, T, 3, heads, head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # each (N, heads, T, head_dim)

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5  # (N, heads, T, T)
        att = att.softmax(dim=-1)            # no mask here (decoder would mask future tokens)
        att = self.dropout(att)

        out = att @ v                        # (N, heads, T, head_dim)
        out = out.transpose(1, 2).reshape(N, T, D)  # concat heads -> (N, T, D)
        return self.proj(out)                # (N, T, D)

## 3. Encoder block
Transformer (Post-LN): `x = LN(x + MHA(x))`, `x = LN(x + FFN(x))`

ViT (Pre-LN): `x = x + MHA(LN(x))`, `x = x + MLP(LN(x))` - the residual path stays clean, so deep stacks train without warmup tricks.

### Formula: LayerNorm
$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta, \qquad \mu = \frac{1}{D}\sum_{i=1}^{D} x_i,\quad \sigma^2 = \frac{1}{D}\sum_{i=1}^{D}(x_i - \mu)^2$$
- $x \in \mathbb{R}^D$: one token's feature vector (normalized per token, not per batch)
- $\mu, \sigma^2$: mean and variance over the $D$ features
- $\epsilon$: small constant for numerical stability ($10^{-5}$)
- $\gamma, \beta \in \mathbb{R}^D$: learned scale and shift
- $\odot$: element-wise product

### Formula: Pre-LN encoder block
$$z'_\ell = z_{\ell-1} + \text{MSA}(\text{LN}(z_{\ell-1})), \qquad z_\ell = z'_\ell + \text{MLP}(\text{LN}(z'_\ell))$$
$$y = \text{LN}(z_L^0)\,W_{head}$$
- $z_{\ell-1}$: input to block $\ell$; $z'_\ell$: after attention sub-layer; $z_\ell$: block output
- MLP: $\text{Linear}(D \to 4D) \to \text{GELU} \to \text{Linear}(4D \to D)$
- $L$: number of blocks (6); $z_L^0$: final `[CLS]` output; $W_{head}$: classifier ($D \times 10$)
- Post-LN (original Transformer) instead does $z = \text{LN}(z + f(z))$, putting LN on the residual path; Pre-LN keeps an identity residual path, which trains more stably

### Formula: GELU
$$\text{GELU}(x) = x\,\Phi(x) \approx 0.5\,x\left(1 + \tanh\!\left[\sqrt{2/\pi}\,\left(x + 0.044715\,x^3\right)\right]\right)$$
- $x$: scalar input (applied element-wise)
- $\Phi(x) = P(X \le x),\ X \sim \mathcal{N}(0,1)$: standard normal CDF
- tanh form: cheap approximation (`nn.GELU()` default uses the exact erf form)

In [ ]:
class EncoderBlock(nn.Module):
    # (N, T, D) -> (N, T, D)
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(            # same position-wise FFN, but GELU instead of ReLU
            nn.Linear(dim, mlp_ratio * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # Pre-LN, (N, T, D)
        x = x + self.mlp(self.ln2(x))   # (N, T, D) -> (N, T, 4D) -> (N, T, D)
        return x

## 4. ViT
- `[CLS]` token: a learned vector prepended to the sequence (borrowed from BERT). After the encoder, its output summarizes the whole image and goes to the classifier.
- Positional embedding: **learned** (Transformer used fixed sin/cos). Without it attention is permutation-invariant and the model can't tell where a patch came from.

In [ ]:
class ViT(nn.Module):
    # (N, 3, 32, 32) -> (N, num_classes)
    def __init__(self, img_size=32, patch_size=4, in_ch=3, num_classes=10,
                 dim=192, depth=6, num_heads=3, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_ch, dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))  # +1 for CLS
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[EncoderBlock(dim, num_heads, mlp_ratio, dropout) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)          # final LN needed with Pre-LN (last block output is un-normalized)
        self.head = nn.Linear(dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x)                        # (N, 64, D)
        cls = self.cls_token.expand(x.shape[0], -1, -1) # (N, 1, D)
        x = torch.cat([cls, x], dim=1)                 # (N, 65, D)
        x = self.dropout(x + self.pos_embed)           # + (1, 65, D) broadcast over N

        x = self.blocks(x)                             # (N, 65, D)
        x = self.ln(x)                                 # (N, 65, D)
        return self.head(x[:, 0])                      # classify from CLS -> (N, num_classes)

model = ViT().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(torch.randn(2, 3, 32, 32, device=device)).shape)

## 5. Data - CIFAR-10
32x32 images, 10 classes, 50k train / 10k test.

ViTs have **no conv inductive bias** (locality, translation equivariance), so on small datasets they overfit fast. Augmentation (random crop + flip) matters much more than for a CNN.

In [ ]:
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.CIFAR10("../data", train=True, download=True, transform=train_tf)
test_ds = datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

## 6. Training
Standard ViT recipe: **AdamW** + weight decay + **linear warmup then cosine decay** (same warmup idea as the original Transformer's LR schedule).

### Formula: Cross-entropy with label smoothing
$$\mathcal{L} = -\sum_{k=1}^{K} q_k \log p_k, \qquad q_k = (1-\epsilon)\,\mathbb{1}[k = y] + \frac{\epsilon}{K}, \qquad p = \text{softmax}(\text{logits})$$
- $K$: number of classes (10)
- $y$: true class index; $\mathbb{1}[\cdot]$: indicator (1 if true, else 0)
- $\epsilon$: smoothing amount (0.1) -> target 0.91 on true class, 0.01 on each other
- $q_k$: smoothed target distribution; $p_k$: predicted probability of class $k$
- discourages over-confident logits (regularizer)

### Formula: AdamW
$$g_t = \nabla_\theta \mathcal{L}(\theta_{t-1})$$
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_t = \theta_{t-1} - \eta_t\left(\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda\,\theta_{t-1}\right)$$
- $\theta_t$: parameters at step $t$; $g_t$: gradient
- $m_t$: EMA of gradients (1st moment, momentum); $v_t$: EMA of squared gradients (2nd moment)
- $\beta_1, \beta_2$: decay rates (defaults 0.9, 0.999)
- $\hat m_t, \hat v_t$: bias-corrected moments (undo the zero-init bias early on)
- $\eta_t$: learning rate (peak $10^{-3}$, set by the schedule below)
- $\epsilon$: stability constant ($10^{-8}$)
- $\lambda$: weight decay (0.05)
- Decoupled: Adam+L2 adds $\lambda\theta$ to $g_t$, so it gets divided by $\sqrt{\hat v_t}$ (weak decay for large-gradient weights); AdamW applies decay directly to $\theta$, uniformly

### Formula: LR schedule: linear warmup + cosine decay
$$\eta_t = \eta \cdot \frac{t}{T_w} \quad (t \le T_w), \qquad \eta_t = \eta_{min} + \tfrac{1}{2}(\eta - \eta_{min})\left(1 + \cos\frac{\pi t'}{T}\right) \quad (t > T_w)$$
- $\eta$: peak LR ($10^{-3}$); $\eta_{min}$: final LR (0, PyTorch default)
- $t$: current optimizer step (scheduler steps per batch)
- $T_w$: warmup steps (3 epochs $\times$ batches/epoch); code starts at $0.01\,\eta$ (`start_factor=0.01`) rather than 0
- $t' = t - T_w$: steps since warmup ended
- $T$: cosine length (27 epochs $\times$ batches/epoch)
- warmup avoids large, noisy updates while Adam's moment estimates are unreliable; cosine anneals smoothly to $\eta_{min}$

In [ ]:
epochs = 30
lr = 1e-3
warmup_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs * len(train_loader))
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(epochs - warmup_epochs) * len(train_loader))
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs * len(train_loader)])
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()  # logits (N, 10) -> preds (N,)
        total += y.size(0)
    return correct / total

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)  # x (N, 3, 32, 32) -> logits (N, 10) vs y (N,) -> scalar

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()  # per step, not per epoch
        total_loss += loss.item()

    print(f"epoch {epoch+1:2d} | loss {total_loss / len(train_loader):.4f} | test acc {evaluate(model, test_loader):.4f}")

## Notes
- Small ViT on CIFAR-10 from scratch lands around ~75-80% in 30 epochs; a ResNet of similar size gets ~90%+. The paper's point: ViT only beats CNNs when pre-trained on huge data (ImageNet-21k / JFT-300M), because it has to *learn* locality that convs get for free.
- Patch size trade-off: smaller patch -> more tokens -> better accuracy but attention cost grows as $O(T^2)$.